In [ ]:
import os
os.system("cd tests/6_5.0/ && python ../../scripts/tvm/main.py")

In [ ]:
from toolbox import manuscriptPlots
from scripts.data_processing import 
import numpy as np
import pandas as pd
import os


In [ ]:
from toolbox import manuscriptPlots
from scripts.data_processing import download
import numpy as np
import pandas as pd
import os

s0_vals = [4.8,4.9,5.0,5.1,5.2,5.3]
d_list = ["1_cell_decrease/", "2_cells/","4_cells/"]
# colormap = {"":"red", "final":"blue"}
# labelmap = {"init": "Initial", "final": "Final"}
# filename = "graphs/stress_tensor_manuscript/kappa_{:.2f}.png".format(p)
for d in d_list:
    for s0 in s0_vals:
        experiment = "{}7_{:.1f}/".format(d,s0)
        download(experiment)
        input_file = "{}error.csv".format(experiment)
        plotter = manuscriptPlots.plot()
        plotter.set_ylim(1e-3,2)
        plotter.set_xlim(0,100)
        # plotter.set_title(" p = {:.2f}".format(p))
        plotter.set_xlabel("Epochs")
        plotter.set_ylabel(r"$<\frac{|\sigma_{target}-\sigma|}{\sigma}>$")
        # plotter.set_ylabel(r"$<\sum_{cells}{|s_{0,cell}^{(A)}-s_{0,cell}^{(B)}|}>$")

        plotter.set_xticks([20*i for i in range(100)])
        plotter.set_yticks([0.05*i for i in range(1,10)])
        plotter.set_yScaled()

        plotter.initialize_figure()
        # plotter.set_yLog()
        df = pd.read_csv(input_file)
        costs = df["mean"].to_numpy()
        err = df["sem"].to_numpy()
        if not len(costs):
            continue
        epochs = [i for i in range(len(costs))]
        plotter.ax.set_yscale("log")
        plotter.plot_xy(epochs,costs,label = r"$\sigma_{trained}>\sigma_{initial}$", color= "red",alpha = 0.8)
        plotter.plot_errorfill(epochs,costs,err)
        # plotter.plot_xy_errorbar([i for i in range(len(costs))],costs, err, label = r"$\sigma_{trained}<\sigma_{initial}$", color= "red",alpha = 0.8)
        plotter.save_fig("/Users/shabeebameen/Projects/tvm-fire/{}error.jpg".format(experiment))

# df = pd.read_csv("../tvm-fire/sp_5_cell_increase/costs.csv")
# costs = df["mean"].to_numpy()
# err = df["sem"].to_numpy()
# plotter.plot_xy_errorbar([i for i in range(len(costs))],costs, err, label = r"$\sigma_{trained}>\sigma_{initial}$", color= "blue",alpha = 0.8)

# df = pd.read_csv("../tvm-fire/mp_2_cells/patternA_costs.csv")
# costs = df["mean"].to_numpy()
# err = df["sem"].to_numpy()
# plotter.plot_xy_errorbar([i for i in range(len(costs))],costs, err, label = "Pattern A", color= "red",alpha = 0.6)

# df = pd.read_csv("../tvm-fire/mp_2_cells/patternB_costs.csv")
# costs = df["mean"].to_numpy()
# err = df["sem"].to_numpy()
# plotter.plot_xy_errorbar([i for i in range(len(costs))],costs, err, label = "Pattern B", color= "blue",alpha = 0.6)

# df = pd.read_csv("../tvm-fire/mp_2_cells/distances.csv")
# costs = df["mean"].to_numpy()
# err = df["sem"].to_numpy()
# plotter.plot_xy_errorbar([i for i in range(len(costs))],costs, err, label = "_Pattern A", color= "black",alpha = 0.6)

# plotter.save_fig("/Users/shabeebameen/Projects/tvm-fire/mp_2_cells_distances.jpg")


In [ ]:
import os


def write_scripts(output_dir):
    lines_sub = []
    lines_sub.append("executable = /home/mameen/examples/singularity_wrapper.sh\n")
    lines_sub.append("arguments  = /home/mameen/examples/ubuntu18_povray_paula.img /home/mameen/{}minimize_config.sh\n".format(output_dir))
    lines_sub.append("transfer_input_files = /home/mameen/scripts/minimize_config.py, /home/mameen/setup.py, /home/mameen/toolbox\n")
    lines_sub.append("should_transfer_files = YES\n")
    lines_sub.append("output     = output.txt\n")
    lines_sub.append("error      = error.txt\n")
    lines_sub.append("log        = log.txt\n")
    lines_sub.append("getenv     = True\n")
    lines_sub.append("request_cpus = 1\n")
    lines_sub.append("request_memory = 100 MB\n")
    lines_sub.append('Requirements = TARGET.vm_name == "its-u20-nfs-20210413" && regexp("CRUSH", TARGET.name)\n')
    lines_sub.append("queue")
    with open("{}minimize_config.sub".format(output_dir), "w") as f:
        for line in lines_sub:
            f.write(line)

    lines_sh = []
    lines_sh.append("#!/bin/bash\n")
    lines_sh.append("source /home/mameen/.bashrc\n")
    lines_sh.append("pip install --user -e .\n")
    lines_sh.append("python /home/mameen/scripts/minimize_config.py /home/mameen/{}\n".format(output_dir))
    with open("{}minimize_config.sh".format(output_dir), "w") as f:
        for line in lines_sh:
            f.write(line)

    



for s0 in [4.8,4.9,5.0,5.1,5.2,5.3]:
    for i in range(100):
        os.system("cp conf_{:.1f} init_homogeneous/7_{:.1f}/{:03d}/conf".format(s0,s0,i))
        output_dir = "init_homogeneous/7_{:.1f}/{:03d}/".format(s0,i)
        write_scripts(output_dir)


In [ ]:
code = """
import os
for s0 in [4.8,4.9,5.0,5.1,5.2,5.3]:
    os.makedirs("init/7_{:.1f}/".format(s0), exist_ok=True)
    success_dirs = []
    for i in range(100):
        output_dir = "init_homogeneous/7_{:.1f}/{:03d}/".format(s0,i)
        if not os.path.isfile("{}minimized.txt".format(output_dir)):
            continue
        success_dirs.append(output_dir)
    for k, dir in enumerate(success_dirs):
        os.system("cp -r {} init/7_{:.1f}/{:03d}/".format(dir,s0,k))

"""
exec(code)

In [ ]:
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox.training import Training
from toolbox.pattern_trainer import train_random_cells, resume_run
from toolbox.periodic import PeriodicTissue
from toolbox import stress
import numpy as np

dir = "tests/6_5.0/"
resume_run(dir,max_iters = 1000)
# sample = PeriodicTissue.from_config(dir,"sample.topo")

# min_instance = Training.periodic_tissue(sample)
# min_instance.set_cpp_executable_dir("/home/shabeeb/Projects/tvm-fire/build/")
# min_instance.minimize_config()
# sample =  PeriodicTissue.from_config(dir,"minimized.txt")
# stresses = [stress.calculate_max_shear_stress(sample,cellID) for cellID in sample.cells_]
# train_random_cells(dir, target_stress = np.mean(stresses), n_cells = 2,cpp_executable_dir = "/home/shabeeb/Projects/tvm-fire/build/")

Resuming runs in directory: tests/6_5.0/
Parameters for training:
cpp_executable_dir: /home/shabeeb/Projects/tvm-fire/build/
tolerance: 1e-08
max iters: 1000
100
Loading configuration from:  0099.bulk.txt
Loading cell parameters from:  0099.cellParameters.input




Starting iteration: 100




Number of modifiable cells: 216


-------------------------------------
Step 1: Evaluate and store the current (free state) areas of hidden (non-target) cells


-------------------------------------


-------------------------------------
Step 2: CLAMPING
-------------------------------------


Clamping iteration 0
Cell: 4, Final Target Stress: 0.2280490924598594 Temporary Target Stress: 0.2415983688972325, Current stress: 0.2221077066419097, s0: 5.0
Cell: 143, Final Target Stress: 0.2280490924598594 Temporary Target Stress: 0.21874561869690506, Current stress: 0.23212868499960693, s0: 5.0
Initialization start ...


Initial F_rms: 0.00505411
FIRE_only specified; skipping overdamping and reducing i

In [3]:
from toolbox.pattern_trainer import resume_run

resume_run(run_dir = "tests/6_5.0/",tolerance = 1e-8, max_iters = 1000, cpp_executable_dir = "/Users/shabeebameen/Projects/tvm-fire/build/")

Resuming runs in directory: tests/6_5.0/
Parameters for training:
cpp_executable_dir: /Users/shabeebameen/Projects/tvm-fire/build/
tolerance: 1e-08
max iters: 1000
459
Loading configuration from:  0458.bulk.txt
Loading cell parameters from:  0458.cellParameters.input




Starting iteration: 459




Number of modifiable cells: 216


-------------------------------------
Step 1: Evaluate and store the current (free state) areas of hidden (non-target) cells


-------------------------------------


-------------------------------------
Step 2: CLAMPING
-------------------------------------


Clamping iteration 0
Cell: 45, Final Target Stress: 0.2439991557581645 Temporary Target Stress: 0.24454931160757096, Current stress: 0.24377368126924792, s0: 5.0
Cell: 14, Final Target Stress: 0.2439991557581645 Temporary Target Stress: 0.24346268213448222, Current stress: 0.24421902275812002, s0: 5.0
Initialization start ...


Initial F_rms: 0.000104038
FIRE_only specified; skipping overdamping and r

In [2]:
from toolbox.pattern_trainer import resume_run

resume_run(run_dir = "tests/6_5.0_4_cells/",tolerance = 1e-8,cpp_executable_dir = "/Users/shabeebameen/Projects/tvm-fire/build/",max_iters = 1000)

Resuming runs in directory: tests/6_5.0_4_cells/
Parameters for training:
cpp_executable_dir: /Users/shabeebameen/Projects/tvm-fire/build/
tolerance: 1e-08
max iters: 1000
1105
Loading configuration from:  1104.bulk.txt
Loading cell parameters from:  1104.cellParameters.input




Starting iteration: 1105




Number of modifiable cells: 216


-------------------------------------
Step 1: Evaluate and store the current (free state) areas of hidden (non-target) cells


-------------------------------------


-------------------------------------
Step 2: CLAMPING
-------------------------------------


Clamping iteration 0
Cell: 32, Final Target Stress: 0.2465506398325429 Temporary Target Stress: 0.24694628481873732, Current stress: 0.24639016773380362, s0: 5.0
Cell: 15, Final Target Stress: 0.2465506398325429 Temporary Target Stress: 0.2464629564281386, Current stress: 0.24658620388723929, s0: 5.0
Cell: 155, Final Target Stress: 0.2465506398325429 Temporary Target Stress: 0.24592487530116

In [ ]:
import os
experiments_list = ["sp_1_cell_increase",
                    "sp_1_cell_decrease",
                    # "sp_2_cell_increase",
                    # "sp_2_cell_decrease",
                    "sp_5_cell_increase",
                    "sp_5_cell_decrease"]
for experiment in experiments_list:
    os.makedirs(experiment, exist_ok=True)
    # os.system("scp mameen@smatter-login.syr.edu:/home/mameen/{}/histogram_data.csv {}".format(experiment,experiment))
    os.system("scp mameen@smatter-login.syr.edu:/home/mameen/{}/costs.csv {}/".format(experiment,experiment))

    # for i in range(20):
    #     dir = experiment+"/"+"run_{}/".format(i)
    #     os.makedirs(dir, exist_ok= True)
    #     os.system("scp mameen@smatter-login.syr.edu:/home/mameen/{}/costs.csv {}/".format(experiment,experiment))


In [ ]:
import os
for i in range(100):
    for file in ["error.txt","output.txt","log.txt"]:
        os.system("rm init_homogeneous/7_5.2/{:03d}/{}".format(i,file))
    os.system("cp conf init_homogeneous/7_5.2/{:03d}/".format(i))
    os.system("cd init_homogeneous/7_5.2/{:03d}/ && chmod +x minimize_config.sh && condor_submit minimize_config.sub".format(i))

In [ ]:
(.381-0.21744816883482376)/.217

In [ ]:
import pandas as pd
import numpy as np
import glob


dir = "two_cell_training/"
initial_stress = pd.read_csv("{}initial_stress.csv".format(dir))
means = []
final_stress_file = glob.glob("{}*.stresses.csv".format(dir))[-1]
for final_stress_file in sorted(glob.glob("{}*.stresses.csv".format(dir))):
    final_stress = pd.read_csv(final_stress_file)
    targets = final_stress["Target"].to_numpy()
    means.append(np.mean(abs((final_stress["Current"].to_numpy() - targets))/final_stress["Target"].to_numpy()))
   

In [ ]:
from toolbox import manuscriptPlots
import glob
def training_plot(dir):
    plotter = manuscriptPlots.plot()
    plotter.set_ylim(0,0.2)
    plotter.set_xlim(0,np.ceil(len(np.loadtxt("{}costs.txt".format(dir)))/10)*10)
    plotter.set_xticks([50*i for i in range(1000)])
    plotter.set_yticks([.1*i for i in range(1,100)])
    plotter.set_xlabel("Epochs")
    plotter.set_ylabel(r"$<\frac{|\sigma_{target}-\sigma|}{\sigma_{target}}>$")
    plotter.set_yScaled()
    plotter.initialize_figure()

    initial_stress = pd.read_csv("{}initial_stress.csv".format(dir))
    targets = pd.read_csv("{}0000.stresses.csv".format(dir))["Target"].to_numpy()

    means = [np.mean(abs((initial_stress["Stress"].to_numpy() - targets))/targets)]

    final_stress_file = glob.glob("{}*.stresses.csv".format(dir))[-1]
    for final_stress_file in sorted(glob.glob("{}*.stresses.csv".format(dir))):
        final_stress = pd.read_csv(final_stress_file)
        targets = final_stress["Target"].to_numpy()
        means.append(np.mean(abs((final_stress["Current"].to_numpy() - targets))/targets))
        iters = np.arange(len(means))
    plotter.plot_xy(iters, means, color = "blue", label = "_final")
    plotter.save_fig("training.jpg".format(dir))

training_plot("two_cell_training/")

In [ ]:
import os
dir = "samples/"
run = "7_0"
os.system("mkdir -p {}".format(dir+run))
os.system("cp init/7_0/conf {}".format(dir+run))
os.system("cd {} && python /Users/shabeebameen/Projects/tvm-fire/scripts/tvm/main.py conf".format(dir+run))

In [ ]:
from toolbox.periodic import PeriodicTissue
from toolbox.patterns import Patterns
import os
dir = "init/10_0/"
file = "minimized.txt"
# cpp_executable_dir = "tvm/build/"
ids = []
tissue = PeriodicTissue.from_config(dir,file)
for cellID,cell in tissue.cells_.items():
    if cell.center_ is None:
        continue
    if cell.crossBoundary_:
        continue
    if cell.center_[2]>3.7 and cell.center_[2]<4.5:
        ids.append(cellID)
tissue.write_cell_collection_vtk(cells_array=ids, filename="layer.vtk")

In [ ]:
import glob
import pandas as pd
import numpy as np
dir = sorted(glob.glob("patternA/*.bulk.txt"))[-1]

# remove the part before the last slash in dir
file = dir.split("/")[-1]
print(dir)
dir = "patternA/"
df = pd.read_csv("{}initial_stress.csv".format(dir))
print(df)
cellID_to_stress = {int(i): float(stress) for i, stress in zip(df["cellID"].to_numpy(), df["Stress"].to_numpy())}
print(cellID_to_stress)
print(np.loadtxt("{}costs.txt".format(dir)))

In [ ]:
## create cross section of a periodic tissue
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob

dir = "7_bidisperse_5_4.9_0.5_increase/"
# dir = "7_mono_0.5_decrease/"
df = pd.read_csv("{}stresses.csv".format(dir))
print(df["CellID"])
training_cell = df["CellID"].to_numpy()[0]
ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
final_iter = max(ids)
file = "{}.bulk.txt".format(final_iter)
sample = PeriodicTissue.from_config(dir,file)
min = FIREminimization.periodic_tissue(sample)
min.load_cell_parameters("cellParameters.{}.input".format(final_iter))

for cellID, cell in sample.cells_.items():
    cell.vtk_scalar_ = cell.s0_
    cell.vtk_scalar_ = stress.calculate_max_shear_stress(sample, cellID)
normal = np.array([1,0,0])
center = sample.cells_[training_cell].center_
print(sample.cells_[training_cell].s0_)
makeSampleCrossSection(sample=min._config, center = center, normal = normal, filename = "cross_section.vtk")
single_cell = sample.extract_cell(training_cell)
makeSampleCrossSection(sample=single_cell, center=center, normal = normal, filename="single_cell_cross_section.vtk")



In [ ]:
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob
dir = "mp_2_cells/run_0/PatternB/"
df = pd.read_csv("{}initial_stress.csv".format(dir))
# print(df["cellID"])
# ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
# ids = sorted(ids)
# final_iter = 33
# for iter in range(final_iter+1):
file = "0000.bulk.txt"
sample = PeriodicTissue.from_config(dir,file)
sample.write_cell_collection_vtk(df["cellID"].to_numpy(),"target_cells_isolated.vtk",use_scalar=False)
# sample.write_cell_collection_vtk([64],"target_cells_isolated.vtk",use_scalar=False)


# min = FIREminimization.periodic_tissue(sample)

# min.load_cell_parameters("cellParameters.{}.input".format(iter))
# for i, row in df.iterrows():
#     cellID = row["CellID"]
#     target_stress = row["Target"]
#     cell = sample.cells_[cellID]
#     # shear = stress.calculate_max_shear_stress(sample, cellID)
#     # cell.max_shear_stress_ = shear
#     # vtk_scalar = abs(cell.max_shear_stress_ - target_stress)/ target_stress
#     for polygonID in cell.polygons_:
#         polygon = sample.polygons_[polygonID]
#         # polygon.vtk_scalar_ = vtk_scalar
# for cellID in df["CellID"].to_numpy():
    # sample.write_cell_collection_vtk(df["CellID"].to_numpy(),"{}.target_cells.vtk".format(iter),use_scalar=True)
    # sample.write_cell_collection_vtk([cellID],"{}single.{}.vtk".format(dir,cellID),use_scalar=False)

